# 📖 Beyond the Cover Analytics: Inteligência de Mercado Editorial com NLP & LLMs
## Desafio Técnico & Negocial A3Data para Editora
---
### Contexto do Desafio:
A editora contratou a **A3Data** para explorar uma base com mais de **3.000.000 de avaliações de livros** e **212.000 títulos**.
- **Gargalo Atual**: 5 analistas dedicados em tempo integral, levando cerca de 3 dias úteis por análise (120 horas-homem).
- **Custo Salarial da Equipe**: R$ 5.000,00 por analista = **R$ 25.000,00/mês** (R$ 300.000,00/ano).
- **Objetivos Centrais**:
  1. Compreender a performance de autores (quantitativa e qualitativamente).
  2. Mapear a performance e percepção de gêneros e categorias literárias.
  3. Identificar leitores ideais para entrevistas qualitativas em profundidade.
  4. Demonstrar a redução de custos e o ROI (*Return on Investment*) viabilizando a realocação da equipe.
- **Diferencial Técnico**: Processamento de Linguagem Natural (**NLP**) e Grandes Modelos de Linguagem (**LLMs**).

In [ ]:
# 1. Importacao dos modulos e configuracao do ambiente
import os
import sys
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Adiciona o diretorio src ao path
sys.path.append(os.path.abspath('../src'))
from business_metrics import ROICalculator, BusinessScenario
from nlp_engine import NLPEngine
from llm_engine import LLMEngine
from data_loader import DataLoader

print('Ambiente e bibliotecas carregadas com sucesso!')

### 2. Engenharia de Dados de Alta Performance com DuckDB
Devido ao volume do dataset (**2,7 GB e 3 milhões de resenhas**), utilizamos o **DuckDB** para realizar consultas analíticas colunares em streaming direto do disco, consumindo menos de 400 MB de memória RAM.

In [ ]:
# Conexao e inspecao dos KPIs gerais
loader = DataLoader(data_dir='../data')
kpis = loader.get_metricas_gerais()

print(f'Total de Livros Catalogados: {kpis["total_livros"]:,}')
print(f'Total de Avaliacoes (Reviews): {kpis["total_avaliacoes"]:,}')
print(f'Total de Leitores Unicos: {kpis["total_usuarios"]:,}')
print(f'Nota Media Global do Catalogo: {kpis["media_score_global"]:.2f} / 5.0')

### 3. Análise Exploratória de Dados (EDA - Exploratory Data Analysis)
Mapeamento dos autores mais avaliados e categorias dominantes no catálogo.

In [ ]:
# Autores mais avaliados na base
top_autores = loader.get_top_autores(limit=15)
df_top_autores = pd.DataFrame(top_autores)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_top_autores, x='total_avaliacoes', y='autor', palette='viridis')
plt.title('Top 15 Autores por Volume de Avaliações')
plt.xlabel('Quantidade de Resenhas')
plt.ylabel('Autor')
plt.tight_layout()
plt.show()

### 4. Processamento de Linguagem Natural (NLP): Sentimentos & Tópicos (TF-IDF)
Aplicamos o algoritmo **VADER** (*Valence Aware Dictionary and sEntiment Reasoner*) para determinar a polaridade das resenhas e **TF-IDF** (*Term Frequency-Inverse Document Frequency*) com n-gramas para identificar termos representativos de cada autor.

In [ ]:
nlp = NLPEngine()

# Amostra de analise de um autor de destaque
autor_estudo = top_autores[0]['autor']
print(f'Analisando autor: {autor_estudo}')

df_autor = loader.get_reviews_por_autor(autor_estudo, limit=100)
df_autor['sentimento'] = df_autor['text'].apply(nlp.analisar_sentimento_resenha)
df_autor['sent_cat'] = df_autor['sentimento'].apply(lambda x: x['categoria'])

print('\nDistribuicao de Sentimento:')
print(df_autor['sent_cat'].value_counts(normalize=True).round(3) * 100)

# Extracao de Topicos TF-IDF
topicos = nlp.extrair_topicos_chave(df_autor['text'].dropna().tolist(), top_n=8)
print('\nTopicos TF-IDF mais relevantes:')
for t in topicos:
    print(f'- {t["termo"]}: {t["score"]}')

### 5. Algoritmo de Seleção de Leitores para Entrevista Qualitativa
O algoritmo pondera **profundidade textual** (tamanho da resenha), **nuance crítica** (notas 2, 3 e 4 com argumentação balanceada) e **qualidade do resumo**, gerando o **Interview Priority Score (0 a 100)**.

In [ ]:
df_ranqueado = nlp.calcular_score_relevancia_entrevista(df_autor)
top_candidatos = df_ranqueado.sort_values(by='interview_priority_score', ascending=False).head(5)

print('Top 5 Candidatos para Entrevista Qualitativa:')
display_cols = ['profileName', 'score', 'word_count', 'interview_priority_score', 'summary']
print(top_candidatos[display_cols].to_string())

### 6. Diferencial com LLM: Geração de Roteiro de Entrevista Personalizado
A partir do perfil do leitor selecionado pelo algoritmo de NLP, o **LLM** gera um roteiro estruturado com perguntas investigativas para os analistas da editora conduzirem uma entrevista de 20 minutos.

In [ ]:
llm = LLMEngine()
leitor_selecionado = top_candidatos.iloc[0]

roteiro = llm.gerar_roteiro_entrevista_usuario(
    usuario_info=leitor_selecionado.to_dict(),
    livro_foco=leitor_selecionado['Title'],
    resenha_leitor=leitor_selecionado['text'],
    sentimento_resenha=leitor_selecionado['sent_cat']
)

print(roteiro)

### 7. Business Case & Análise de ROI
Demonstração quantitativa da economia de custos e realocação da equipe de analistas.

In [ ]:
calc = ROICalculator()
print(calc.gerar_relatorio_impacto_executivo())